In [3]:
sent1 = 'It is good for our progress'
sent2 = 'They have decided that it was not good'

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

In [6]:
cvt = CountVectorizer()

In [7]:
vect = cvt.fit_transform([sent1,sent2])

In [8]:
import pandas as pd

In [10]:
df = pd.DataFrame(vect.toarray(), columns=cvt.get_feature_names_out())
df

,decided,for,good,have,is,it,not,our,progress,that,they,was
0,0,1,1,0,1,1,0,1,1,0,0,0
1,1,0,1,1,0,1,1,0,0,1,1,1


###Continuous bag of words

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [12]:
lines = ['It was a nice rainy day.','The things are so beutiful in his point',
        'When your focus is clear,you won.','Many many happy returns of the day.']

In [14]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(lines)

In [17]:
tokenizer.word_counts

OrderedDict([('it', 1),
             ('was', 1),
             ('a', 1),
             ('nice', 1),
             ('rainy', 1),
             ('day', 2),
             ('the', 2),
             ('things', 1),
             ('are', 1),
             ('so', 1),
             ('beutiful', 1),
             ('in', 1),
             ('his', 1),
             ('point', 1),
             ('when', 1),
             ('your', 1),
             ('focus', 1),
             ('is', 1),
             ('clear', 1),
             ('you', 1),
             ('won', 1),
             ('many', 2),
             ('happy', 1),
             ('returns', 1),
             ('of', 1)])

In [18]:
tokenizer.word_index

{'day': 1,
 'the': 2,
 'many': 3,
 'it': 4,
 'was': 5,
 'a': 6,
 'nice': 7,
 'rainy': 8,
 'things': 9,
 'are': 10,
 'so': 11,
 'beutiful': 12,
 'in': 13,
 'his': 14,
 'point': 15,
 'when': 16,
 'your': 17,
 'focus': 18,
 'is': 19,
 'clear': 20,
 'you': 21,
 'won': 22,
 'happy': 23,
 'returns': 24,
 'of': 25}

In [19]:
mat = tokenizer.texts_to_matrix(lines)
mat

array([[0., 1., 0., 0., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        1., 1., 1., 1., 1., 1., 1., 0., 0., 0.],
       [0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 1., 1., 1.]])

In [20]:
seq = tokenizer.texts_to_sequences(lines)
seq

[[4, 5, 6, 7, 8, 1],
 [2, 9, 10, 11, 12, 13, 14, 15],
 [16, 17, 18, 19, 20, 21, 22],
 [3, 3, 23, 24, 25, 2, 1]]

In [21]:
padded = pad_sequences(seq, maxlen=10, padding='pre')
padded

array([[ 0,  0,  0,  0,  4,  5,  6,  7,  8,  1],
       [ 0,  0,  2,  9, 10, 11, 12, 13, 14, 15],
       [ 0,  0,  0, 16, 17, 18, 19, 20, 21, 22],
       [ 0,  0,  0,  3,  3, 23, 24, 25,  2,  1]], dtype=int32)

In [22]:
#dataset twitter sentiments

In [23]:
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split
import re
import numpy as np

In [24]:
###Data preparation

In [25]:
data = pd.read_csv("twitter_sentiments.csv", names = ['id','loc','label','text'])

In [26]:
data

,id,loc,label,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
...,...,...,...,...
74677,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74678,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74679,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74680,9200,Nvidia,Positive,Just realized between the windows partition of...


In [27]:
data.shape

(74682, 4)

In [29]:
data.dtypes

id        int64
loc      object
label    object
text     object
dtype: object

In [33]:
data['text'] = data['text'].astype(str) 

In [38]:
#Text Cleaning
def clean_text(text):
    text = text.lower()  #lowercase
    text = re.sub(r"[^a-zA-z]+"," ", text)  #remove non-alphanumeric characters
    return text

In [39]:
clean_text("Hello friends! How are you???? Welcome... 627851!!!!")

'hello friends how are you welcome '

In [40]:
data["text"] = data["text"].apply(clean_text)

In [41]:
data

,id,loc,label,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,i am coming to the borders and i will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
...,...,...,...,...
74677,9200,Nvidia,Positive,just realized that the windows partition of my...
74678,9200,Nvidia,Positive,just realized that my mac window partition is ...
74679,9200,Nvidia,Positive,just realized the windows partition of my mac ...
74680,9200,Nvidia,Positive,just realized between the windows partition of...


In [42]:
#Feature and target preparation
comments = data["text"].tolist() #input variable
targets = data['label'].values  #output variable

In [44]:
pd.DataFrame(targets).value_counts()

0         
Negative      22542
Positive      20832
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

In [45]:
#### b.Generate training data

In [46]:
#Tokenization and padding 
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(comments)
sequences = tokenizer.texts_to_sequences(comments)
padded_sequences = pad_sequences(sequences, maxlen=200)

In [47]:
padded_sequences.shape

(74682, 200)

In [48]:
padded_sequences

array([[   0,    0,    0, ..., 1695,   12,   26],
       [   0,    0,    0, ...,  424,   12,   26],
       [   0,    0,    0, ...,  424,   12,   26],
       ...,
       [   0,    0,    0, ...,  302,   15, 2055],
       [   0,    0,    0, ...,  302,   15, 2055],
       [   0,    0,    0, ...,  302,   15, 2055]], dtype=int32)

In [50]:
#output data preparation
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(targets)

from keras.utils import to_categorical
y_new = to_categorical(y)

In [51]:
y_new

array([[0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       ...,
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.]])

In [52]:
#Train-test split / Cross validation
x_train, x_test, y_train, y_test = train_test_split(
    padded_sequences, y_new, test_size=0.2, random_state=0)

In [53]:
x_train.shape

(59745, 200)

In [54]:
x_test.shape

(14937, 200)

c.Train model

In [57]:
#Model definition (customize architecture as needed)
model = Sequential()
model.add(Embedding(5000, 128, input_length=200))
model.add(LSTM(64))
model.add(Dense(4, activation='softmax'))
#Multi-Label output with sigmoid activation

In [59]:
#Model Compilation
model.compile(loss="categorical_crossentropy",
             optimizer="adam",metrics=["accuracy"])

In [60]:
#Model training
model.fit(x_train, y_train, epochs=3, batch_size=32,
         validation_data=(x_test, y_test))

Epoch 1/3
1868/1868 ━━━━━━━━━━━━━━━━━━━━ 69s 36ms/step - accuracy: 0.6171 - loss: 0.9325 - val_accuracy: 0.7123 - val_loss: 0.7419
Epoch 2/3
1868/1868 ━━━━━━━━━━━━━━━━━━━━ 68s 36ms/step - accuracy: 0.7705 - loss: 0.6004 - val_accuracy: 0.7620 - val_loss: 0.6198
Epoch 3/3
1868/1868 ━━━━━━━━━━━━━━━━━━━━ 68s 36ms/step - accuracy: 0.8318 - loss: 0.4472 - val_accuracy: 0.7982 - val_loss: 0.5437


D.output

In [77]:
new_comment = "I love nature."
new_sequence = tokenizer.texts_to_sequences([clean_text(new_comment)])
padded_new_sequence = pad_sequences(new_sequence, maxlen=200)
prediction = model.predict(padded_new_sequence)[0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step


In [78]:
prediction

array([0.04002073, 0.05715077, 0.16657226, 0.7362562 ], dtype=float32)

In [79]:
le.inverse_transform([np.argmax(prediction)])[0]

'Positive'